## Shoreline data collection for specified site (Nha Mat to Vinh Chau in the Lower Mekong Data) for shoreline change rate and yearly shoreline position

This notebook downloads the Shoreline Monitor v2.1 data for the study area in the Mekong Delta (Vinh Chau – Nha Mat): the yearly shoreline positions per transect, the shoreline vectors, and the published change rates (GCTR).

Requires an access token (SAS token), from Deltares Shoreline Monitor

In [1]:
import os
from pathlib import Path
import pandas as pd
import geopandas as gpd
import pystac
from dotenv import load_dotenv
import coastpy
from coastpy.utils.config import fetch_sas_token

In [2]:
# Load in SAS token, access needs to be gained before able to run this script

load_dotenv()

# Get SAS token, insert SAS token from Deltares
sas_token = os.getenv("AZURE_STORAGE_SAS_TOKEN")
sas_token = fetch_sas_token(sas_token)

# Connect to CoastPy catalogue
STAC_VERSION = "v2.1"     # was "v1", replace with newest version

coclico_catalog = pystac.Catalog.from_file(
    f"https://coclico.blob.core.windows.net/stac/{STAC_VERSION}/catalog.json"
)
print("Collections:", [child.id for child in coclico_catalog.get_children()])

Collections: ['coastal-grid', 'coastal-zone', 'deltares-delta-dtm', 'gcts', 'global-coastal-typology', 'overture-buildings', 's2-l2a-composite', 'shoreline-projections', 'shoreline-projections-edito', 'shorelinemonitor-shorelines', 'shorelinemonitor-series', 'gctr']


In [3]:
# Mekong Delta study area: Nha Mat – Vinh Chau, define boxes for data collection in specific area
west = 105.3
south = 8.9
east = 106.4
north = 9.8

In [4]:
# CoastPy datasets

collection_change_rate = coclico_catalog.get_child("gctr")
collection_series = coclico_catalog.get_child("shorelinemonitor-series")
collection_shorelines = coclico_catalog.get_child("shorelinemonitor-shorelines")


# Query engines

db_change_rate = coastpy.io.STACQueryEngine(
    stac_collection=collection_change_rate,
    storage_backend="azure"
)

db_series = coastpy.io.STACQueryEngine(
    stac_collection=collection_series,
    storage_backend="azure"
)

db_shorelines = coastpy.io.STACQueryEngine(
    stac_collection=collection_shorelines,
    storage_backend="azure"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
# Folder where all shoreline data are stored

data_folder = Path("../data")

data_folder.mkdir(
    parents=True,
    exist_ok=True
)


# Individual datasets
data_file_change_rate = data_folder / f"mekong_change_rate_{STAC_VERSION}.parquet"
data_file_series      = data_folder / f"mekong_series_{STAC_VERSION}.parquet"
data_file_shorelines  = data_folder / f"mekong_shorelines_{STAC_VERSION}.parquet"

In [15]:
# CHANGE RATE DATA

if data_file_change_rate.exists():

    print("Loading saved change-rate data...")

    change_rate = gpd.read_parquet(
        data_file_change_rate
    )

else:

    print("Downloading change-rate data from CoastPy...")

    change_rate = db_change_rate.get_data_within_bbox(
        west,
        south,
        east,
        north,
        sas_token=sas_token
    )

    change_rate.to_parquet(
        data_file_change_rate
    )

    print("Change-rate data saved!")


print("Change rate shape:", change_rate.shape)

Loading saved change-rate data...
Change rate shape: (1547, 80)


In [16]:
# SHORELINE TIME-SERIES DATA

if data_file_series.exists():

    print("Loading saved series data...")

    series = gpd.read_parquet(
        data_file_series
    )

else:

    print("Downloading shoreline series data from CoastPy...")

    series = db_series.get_data_within_bbox(
        west,
        south,
        east,
        north,
        sas_token=sas_token
    )

    series.to_parquet(
        data_file_series
    )

    print("Series data saved!")


print("Series shape:", series.shape)




Loading saved series data...
Series shape: (45680, 54)


In [17]:
years = pd.to_datetime(series["datetime"]).dt.year
print("Series years:", years.min(), "–", years.max())
print(years.value_counts().sort_index().tail(5))

Series years: 1987 – 2025
datetime
2021    1328
2022    1593
2023    1203
2024    1208
2025    1198
Name: count, dtype: int64


In [18]:
# HISTORICAL SHORELINE DATA

if data_file_shorelines.exists():

    print("Loading saved shoreline data...")

    shorelines = gpd.read_parquet(
        data_file_shorelines
    )

else:

    print("Downloading historical shoreline data from CoastPy...")

    shorelines = db_shorelines.get_data_within_bbox(
        west,
        south,
        east,
        north,
        sas_token=sas_token
    )

    shorelines.to_parquet(
        data_file_shorelines
    )

    print("Shoreline data saved!")


print("Shorelines shape:", shorelines.shape)

Loading saved shoreline data...
Shorelines shape: (7584, 21)


In [13]:
# Check that all three datasets are available

print("DATA COLLECTION COMPLETE")
print()

print("Change rate:")
print(change_rate.shape)
print(data_file_change_rate)

print()

print("Series:")
print(series.shape)
print(data_file_series)

print()

print("Shorelines:")
print(shorelines.shape)
print(data_file_shorelines)

DATA COLLECTION COMPLETE

Change rate:
(1547, 80)
../data/mekong_change_rate_v2.1.parquet

Series:
(45680, 54)
../data/mekong_series_v2.1.parquet

Shorelines:
(7584, 21)
../data/mekong_shorelines_v2.1.parquet


## Check whether new data updates are available from the Deltares Shoreline Monitor

In [14]:
# Check for newer shoreline monitor data available (does not overwrite saved files)


def collection_info(catalog, name):
    """Version and time range of a collection as published in the STAC catalogue."""
    collection = catalog.get_child(name)
    start, end = collection.extent.temporal.intervals[0]
    return {
        "collection": name,
        "version": collection.extra_fields.get("version"),
        "start": start,
        "end": end,
    }


def list_releases(container, sas_token, account="coclico"):
    """Release folders in Deltares' storage (may be newer than the catalogue)."""
    try:
        import adlfs
        fs = adlfs.AzureBlobFileSystem(account_name=account, sas_token=sas_token)
        return sorted(path.split("/")[-1] for path in fs.ls(f"{container}/release"))
    except Exception as error:
        return f"could not list releases ({type(error).__name__})"


def latest_year_online(db, bbox, sas_token):
    """Download a small test area and return the most recent year in it."""
    data = db.get_data_within_bbox(*bbox, sas_token=sas_token)
    years = pd.to_datetime(data["datetime"]).dt.year
    return int(years.max()), years.value_counts().sort_index()


def latest_year_saved(path):
    """Most recent year in the parquet file saved earlier."""
    saved = gpd.read_parquet(path)
    return int(pd.to_datetime(saved["datetime"]).dt.year.max())